# Demonstrating SQL on Pandas DataFrames using DuckDB
This notebook demonstrates how to perform common SQL operations on a pandas DataFrame using DuckDB,
comparing native pandas syntax with its equivalent SQL syntax.

In [ ]:
import pandas as pd
import duckdb


In [ ]:
# Load the Boston housing dataset
url = "http://ddc-datascience.s3.amazonaws.com/boston.csv"
url


In [ ]:
df = pd.read_csv(url, index_col=0)
df.shape


## 1. SELECT *
Select all columns and all rows from the dataset.

In [ ]:
# Pandas
df

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
''').df()

## 2. SELECT - One Column
Select only the `med_home_value` column.

In [ ]:
# Pandas
df[['med_home_value']]

In [ ]:
# SQL
duckdb.query('''
  SELECT med_home_value
  FROM df
''').df()

## 3. SELECT - Multiple Columns
Select the `num_rooms`, `prop_tax`, and `med_home_value` columns.

In [ ]:
# Pandas
df[['num_rooms', 'prop_tax', 'med_home_value', ]]

In [ ]:
# SQL
duckdb.query('''
  SELECT num_rooms, prop_tax, med_home_value
  FROM df
''').df()

## 4. WHERE - Single Conditional (Numeric)
Filter rows where the number of rooms (`num_rooms`) is greater than 8.

In [ ]:
# Pandas
df[df['num_rooms'] > 8]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE num_rooms > 8
''').df()

## 5. WHERE - Single Conditional (Binary/Flag)
Filter rows where the property limits the Charles River (`river_front` equals 1).

In [ ]:
# Pandas
df[df['river_front'] == 1]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE river_front = 1
''').df()

## 6. WHERE - Two Conditionals with AND
Filter rows where `num_rooms` is greater than 7 AND `med_home_value` is greater than 40.

In [ ]:
# Pandas
df[(df['num_rooms'] > 7) & (df['med_home_value'] > 40)]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE
      num_rooms > 7
    AND
      med_home_value > 40
''').df()

## 7. WHERE - Two Conditionals with OR
Filter rows where `per_cap_crime` is extremely high (> 50) OR `perc_lower_status` is extremely high (> 30).

In [ ]:
# Pandas
df[(df['per_cap_crime'] > 50) | (df['perc_lower_status'] > 30)]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE
      per_cap_crime > 50
    OR
      perc_lower_status > 30
''').df()

## 8. WHERE - Mixed Conditionals (AND + OR)
Filter rows next to the river where either the crime rate is low (< 0.1) OR rooms are many (> 7).

In [ ]:
# Pandas
df[(df['river_front'] == 1) & ((df['per_cap_crime'] < 0.1) | (df['num_rooms'] > 7))]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE
      river_front = 1
    AND
      ( (per_cap_crime < 0.1) OR (num_rooms > 7) )
''').df()

## 9. WHERE - BETWEEN
Filter rows where `med_home_value` is BETWEEN 25 and 30 (inclusive).

In [ ]:
# Pandas
df[df['med_home_value'].between(25, 30)]

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE
    med_home_value BETWEEN 25 AND 30
''').df()

## 10. LIMIT
Select the first 5 rows of the dataset.

In [ ]:
# Pandas
df.head(5)

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  LIMIT 5
''').df()

## 11. WHERE with LIMIT
Select the first 3 rows where `distance_to_hwy` is 24.

In [ ]:
# Pandas
df[df['distance_to_hwy'] == 24].head(3)

In [ ]:
# SQL
duckdb.query('''
  SELECT *
  FROM df
  WHERE distance_to_hwy = 24
  LIMIT 3
''').df()

## 12. GROUP BY - COUNT
Count the number of properties grouped by their accessibility index to highways (`distance_to_hwy`).

In [ ]:
# Pandas
df.groupby('distance_to_hwy').size().reset_index(name='count')

In [ ]:
# SQL
duckdb.query('''
  SELECT distance_to_hwy, COUNT(*) as count
  FROM df
  GROUP BY distance_to_hwy
''').df()

## 13. GROUP BY - AVG
Find the average home value (`med_home_value`) grouped by whether they are on the river front (`river_front`).

In [ ]:
# Pandas
df.groupby('river_front')['med_home_value'].mean().reset_index(name='avg_value')

In [ ]:
# SQL
duckdb.query('''
  SELECT river_front, AVG(med_home_value) as avg_value
  FROM df
  GROUP BY river_front
''').df()

## 14. GROUP BY - SUM
Find the total proportion of non-retail business acres (`business_prop`) grouped by highway accessibility.

In [ ]:
# Pandas
df.groupby('distance_to_hwy')['business_prop'].sum().reset_index(name='total_business_prop')

In [ ]:
# SQL
duckdb.query('''
  SELECT distance_to_hwy, SUM(business_prop) as total_business_prop
  FROM df
  GROUP BY distance_to_hwy
''').df()

## 15. GROUP BY - MIN & MAX
Find the minimum and maximum room counts grouped by highway accessibility.

In [ ]:
# Pandas
df.groupby('distance_to_hwy')['num_rooms'].agg(min_rooms='min', max_rooms='max').reset_index()

In [ ]:
# SQL
duckdb.query("""
    SELECT
      distance_to_hwy,
      MIN(num_rooms) as min_rooms,
      MAX(num_rooms) as max_rooms
    FROM df
    GROUP BY distance_to_hwy
""").df()

## 16. GROUP BY - Multiple Columns
Group by both `river_front` and `distance_to_hwy` to get the average `nitric_oxide_conc`.

In [ ]:
# Pandas
df.groupby(['river_front', 'distance_to_hwy'])['nitric_oxide_conc'].mean().reset_index(name='avg_nox')

In [ ]:
# SQL
duckdb.query("""
    SELECT river_front, distance_to_hwy, AVG(nitric_oxide_conc) as avg_nox
    FROM df
    GROUP BY river_front, distance_to_hwy
    -- GROUP BY all  --- specific to duckdb
""").df()

## 17. HAVING - Filter Group COUNT
Group by `distance_to_hwy` and only return groups that have more than 30 properties.

In [ ]:
# Pandas
grouped = df.groupby('distance_to_hwy').size().reset_index(name='count')
grouped[grouped['count'] > 30]

In [ ]:
# SQL
duckdb.query("""
    SELECT distance_to_hwy, COUNT(*) as count
    FROM df
    GROUP BY distance_to_hwy
    HAVING COUNT(*) > 30
""").df()

## 18. HAVING - Filter Group AVG
Group by `distance_to_hwy` and only return groups where the average `med_home_value` is greater than 25.

In [ ]:
# Pandas
grouped = df.groupby('distance_to_hwy')['med_home_value'].mean().reset_index(name='avg_value')
grouped[grouped['avg_value'] > 25]

In [ ]:
# SQL
duckdb.query("""
    SELECT distance_to_hwy, AVG(med_home_value) as avg_value
    FROM df
    GROUP BY distance_to_hwy
    HAVING AVG(med_home_value) > 25
""").df()

## 19. HAVING - Filter Group MIN
Group by `river_front` and only return groups where the absolute cheapest home (`MIN(med_home_value)`) is greater than 6.

In [ ]:
# Pandas
grouped = df.groupby('river_front')['med_home_value'].min().reset_index(name='min_value')
grouped[grouped['min_value'] > 6]

In [ ]:
# SQL
duckdb.query("""
    SELECT river_front, MIN(med_home_value) as min_value
    FROM df
    GROUP BY river_front
    HAVING MIN(med_home_value) > 6
""").df()

## 20. Complex Combination (SELECT, WHERE, GROUP BY, HAVING, LIMIT)
Select `distance_to_hwy` and average `med_home_value` for properties built before 1940 (`units_before_1940` > 80), group by highway distance, keep groups with an average value > 15, and limit to the top 3 results.

In [ ]:
# Pandas
filtered_df = df[df['units_before_1940'] > 80]
grouped = filtered_df.groupby('distance_to_hwy')['med_home_value'].mean().reset_index(name='avg_value')
result = grouped[grouped['avg_value'] > 15].head(3)
result

In [ ]:
# SQL
duckdb.query("""
    SELECT distance_to_hwy, AVG(med_home_value) as avg_value
    FROM df
    WHERE units_before_1940 > 80
    GROUP BY distance_to_hwy
    HAVING AVG(med_home_value) > 15
    LIMIT 3
""").df()